# LoRA / QLoRA fine-tune on a free T4

Fine-tune a small open model with parameter-efficient adapters on one free
Kaggle or Colab GPU.

Suggested base models:

- `Qwen/Qwen2.5-1.5B-Instruct` (default)
- `Qwen/Qwen2.5-3B-Instruct` (slower, tighter on a 16 GB T4)
- `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`

> **Honest scope.** This is LoRA/QLoRA fine-tuning of a 1.5B model, not
> pretraining and **not training DeepSeek V4.1 from scratch**. A free T4 cannot
> train a frontier model. The notebook produces a small adapter, optionally
> merged weights, and optionally a Hub repo.

No API key is stored in this notebook. The optional push cell reads `HF_TOKEN`
from notebook secrets only.


## 1. Fail fast if there is no GPU

The rest of the notebook assumes a CUDA GPU. It stops here rather than silently
training on CPU for hours.


In [ ]:
import json
import os
from pathlib import Path

try:
    import torch
except Exception as exc:
    raise SystemExit("PyTorch is not installed. Run the optional install cell first.") from exc

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU is visible. This notebook is built for a free Kaggle/Colab T4. "
        "Choose a GPU runtime, restart the session, and run this cell again. "
        "CPU fine-tuning is deliberately not supported here."
    )

props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
print("gpu:", props.name)
print(f"vram: {vram_gib:.1f} GiB")
print("compute capability:", f"{props.major}.{props.minor}")
print("torch:", torch.__version__)

if vram_gib < 14:
    print("WARNING: less than 14 GiB VRAM. Prefer the 0.5B/1.5B model, lower MAX_SEQ_LEN, or a smaller batch.")


## 2. Optional dependency install

Set `INSTALL_DEPENDENCIES = True` and run this cell only if the imports in the
next cells fail. A free T4 image usually has torch and transformers already.

This is fine-tuning, not full training. `peft` supplies LoRA, `trl` supplies the
supervised fine-tuning loop, `bitsandbytes` supplies 4-bit QLoRA.


In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.46",
            "peft>=0.13",
            "trl>=0.12",
            "datasets>=3",
            "accelerate>=1.0",
            "bitsandbytes>=0.44",
        ]
    )
    print("installed transformers, peft, trl, datasets, accelerate and bitsandbytes")
else:
    print("install skipped; set INSTALL_DEPENDENCIES = True only if imports fail")


## 3. Configuration

These defaults fit a 16 GB T4 with QLoRA:

- 1.5B base model
- 4-bit NF4 quantization when bitsandbytes is available
- sequence length 1024
- batch size 1 with 8 gradient-accumulation steps (effective batch 8)
- one epoch, cosine schedule, paged AdamW 8-bit

The 3B model usually needs `MAX_SEQ_LEN=512` or a smaller effective batch on a
16 GB card.


In [ ]:
MODEL_ID = os.environ.get("FINETUNE_MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")
# Alternative: "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# Alternative: "Qwen/Qwen2.5-3B-Instruct" (reduce MAX_SEQ_LEN on a 16 GB T4)

OUTPUT_DIR = Path(os.environ.get("FINETUNE_OUTPUT_DIR", "lora-adapter"))
DATASET_PATH = Path(os.environ.get("FINETUNE_DATASET", "train.jsonl"))

MAX_SEQ_LEN = int(os.environ.get("FINETUNE_MAX_SEQ_LEN", "1024"))
USE_4BIT = True

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
SEED = 42

# Optional stages. Keep both False until the plain training run works.
MERGE_ADAPTER = False
PUSH_TO_HUB = False
HUB_REPO_ID = os.environ.get("FINETUNE_HUB_REPO", "")

print("model:", MODEL_ID)
print("dataset:", DATASET_PATH.resolve())
print("adapter output:", OUTPUT_DIR.resolve())
print("max_seq_len:", MAX_SEQ_LEN)
print("use_4bit:", USE_4BIT)
print("effective batch:", BATCH_SIZE * GRAD_ACCUM)


## 4. Dataset path and format

The cell below accepts JSONL with any of these shapes:

```json
{"text": "full training text"}
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
{"instruction": "...", "input": "...", "output": "..."}
```

If `train.jsonl` does not exist, the cell writes a two-row example so the wiring
can be tested before you point it at a real dataset. Replace it with your own
data.


In [ ]:
def show_dataset_example(path):
    if not path.exists():
        print(f"{path} does not exist; writing a tiny example file")
        examples = [
            {
                "messages": [
                    {"role": "user", "content": "What is 2 + 2?"},
                    {"role": "assistant", "content": "4"},
                ]
            },
            {
                "text": "### Question:\nName the capital of Nigeria.\n### Answer:\nAbuja"
            },
        ]
        path.write_text(
            "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in examples),
            encoding="utf-8",
        )

    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle, 1):
            if index > 3:
                break
            print(index, line.strip()[:600])


show_dataset_example(DATASET_PATH)


## 5. Load the base model

QLoRA keeps the base weights in 4-bit NF4 and trains small fp16 adapters. On a
T4 use fp16, not bf16: T4 (Turing) has no native bf16.

If `bitsandbytes` is missing, the cell falls back to fp16 LoRA. That uses more
VRAM and may fail for a 3B model; prefer fixing bitsandbytes first.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = None
if USE_4BIT:
    try:
        import bitsandbytes  # noqa: F401

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        print("using 4-bit NF4 quantization")
    except Exception as exc:
        print("bitsandbytes unavailable; falling back to fp16 LoRA:", exc)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device)
print("dtype:", next(model.parameters()).dtype)
print("4-bit:", bnb_config is not None)


## 6. Load and format the dataset

Rows are normalised to a single `text` column. Chat-style rows go through the
tokenizer's chat template so the model sees the format it was trained on.


In [ ]:
from datasets import load_dataset

if not DATASET_PATH.exists():
    raise SystemExit(f"dataset not found: {DATASET_PATH}")

raw = load_dataset("json", data_files=str(DATASET_PATH), split="train")


def to_text(example):
    if isinstance(example.get("text"), str) and example["text"].strip():
        return {"text": example["text"].strip()}

    messages = example.get("messages")
    if isinstance(messages, list) and messages:
        if getattr(tokenizer, "chat_template", None):
            return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}
        parts = []
        for message in messages:
            role = message.get("role", "user")
            content = message.get("content", "")
            parts.append(f"<|{role}|>\n{content}")
        return {"text": "\n".join(parts)}

    instruction = example.get("instruction")
    output = example.get("output")
    if isinstance(instruction, str) and isinstance(output, str):
        input_text = example.get("input") or ""
        if getattr(tokenizer, "chat_template", None):
            return {
                "text": tokenizer.apply_chat_template(
                    [
                        {"role": "user", "content": f"{instruction}\n\n{input_text}".strip()},
                        {"role": "assistant", "content": output},
                    ],
                    tokenize=False,
                )
            }
        return {
            "text": (
                f"### Instruction:\n{instruction}\n\n"
                f"### Input:\n{input_text}\n\n"
                f"### Output:\n{output}"
            )
        }

    return {"text": None}


formatted = raw.map(to_text, remove_columns=raw.column_names, desc="formatting")
formatted = formatted.filter(
    lambda row: isinstance(row["text"], str) and len(row["text"].strip()) > 0
)

print(formatted)
print("rows:", len(formatted))
if len(formatted) == 0:
    raise SystemExit(
        "No usable rows. Provide {text}, {messages} or {instruction,input,output} per line."
    )
print("first row preview:\n", formatted[0]["text"][:600])


## 7. Attach LoRA adapters and build the trainer

Only the adapter weights are trainable. The base model stays frozen (and, with
QLoRA, 4-bit quantized). `prepare_model_for_kbit_training` enables the gradient
checkpointing and casting that quantized training needs.

`SFTTrainer`'s keyword names have changed across TRL versions, so the cell
inspects the installed signature instead of pinning one API shape.


In [ ]:
import inspect

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments
from trl import SFTTrainer

if bnb_config is not None:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit" if bnb_config is not None else "adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,
)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": formatted,
    "dataset_text_field": "text",
}

signature = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in signature:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

if "max_length" in signature:
    trainer_kwargs["max_length"] = MAX_SEQ_LEN
else:
    trainer_kwargs["max_seq_length"] = MAX_SEQ_LEN

try:
    trainer = SFTTrainer(**trainer_kwargs)
except TypeError as exc:
    # Some TRL releases removed dataset_text_field and default to "text".
    if "dataset_text_field" in trainer_kwargs:
        trainer_kwargs.pop("dataset_text_field")
        trainer = SFTTrainer(**trainer_kwargs)
    else:
        raise exc

trainer.model.print_trainable_parameters()
print("trainer ready")


## 8. Train

One epoch on a small dataset is the sane first run. Watch the loss and the
clock; if a run is going to exceed the free session, reduce rows or sequence
length rather than hoping the runtime survives.

Checkpoints are written under `lora-adapter/checkpoints` every 50 steps.


In [ ]:
train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
trainer.save_state()

metrics = dict(train_result.metrics)
metrics["train_rows"] = len(formatted)
metrics["model_id"] = MODEL_ID
metrics["max_seq_len"] = MAX_SEQ_LEN
Path("train-metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(json.dumps(metrics, indent=2))
print("adapter:", OUTPUT_DIR.resolve())


## 9. Merge the adapter (optional)

Merging writes a standalone fp16 model. It needs enough VRAM to hold the base
model outside 4-bit. On a 16 GB T4 a 1.5B merge fits; a 3B merge is tight.

Leave `MERGE_ADAPTER = False` to keep only the small adapter. The adapter is
usually the better artifact to move between machines.


In [ ]:
if MERGE_ADAPTER:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM

    required_gib = 8.0
    if torch.cuda.get_device_properties(0).total_memory / 1024**3 < required_gib:
        raise SystemExit(f"Need at least {required_gib:.1f} GiB VRAM to merge; keep the adapter instead.")

    if "model" in globals():
        del model
    torch.cuda.empty_cache()

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    merged = PeftModel.from_pretrained(base, str(OUTPUT_DIR))
    merged = merged.merge_and_unload()

    merged_dir = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "-merged")
    merged.save_pretrained(str(merged_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(merged_dir))
    model = merged
    print("merged model:", merged_dir.resolve())
else:
    print("MERGE_ADAPTER is False; keeping adapter only at", OUTPUT_DIR.resolve())


## 10. Push to Hugging Face (optional, owner-gated)

This cell does nothing unless **all** of these are true:

1. `PUSH_TO_HUB = True`
2. `HUB_REPO_ID` is set
3. `HF_TOKEN` exists in notebook secrets (Colab Secrets or Kaggle Secrets) or
   the process environment

The token is never printed and is never written into the notebook. Do not paste
a token into a cell. Create a fine-grained write token and store it as a
notebook secret.


In [ ]:
def read_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return None


if PUSH_TO_HUB:
    if not HUB_REPO_ID:
        raise SystemExit("Set HUB_REPO_ID before pushing.")
    token = read_hf_token()
    if not token:
        raise SystemExit(
            "HF_TOKEN is not set in notebook secrets or the environment. "
            "Refusing to push. Add it as a notebook secret, not in a cell."
        )

    from huggingface_hub import HfApi

    api = HfApi(token=token)
    api.create_repo(repo_id=HUB_REPO_ID, private=True, exist_ok=True)
    trainer.model.push_to_hub(HUB_REPO_ID, private=True, token=token)
    tokenizer.push_to_hub(HUB_REPO_ID, private=True, token=token)
    print("pushed adapter to", HUB_REPO_ID)
else:
    print("PUSH_TO_HUB is False; nothing uploaded")


## 11. Smoke-test the fine-tuned model

Greedy decoding keeps the sample repeatable. This is a wiring check, not an
evaluation. A proper evaluation needs a held-out set and task-specific metrics.


In [ ]:
if "model" in globals() and "tokenizer" in globals():
    model.eval()
    messages = [{"role": "user", "content": "Explain LoRA in one sentence."}]
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = "User: Explain LoRA in one sentence.\nAssistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(next(model.parameters()).device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    print(tokenizer.decode(output[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True))
else:
    print("Run the model and training cells first.")


## Scope, limits and security

- **Fine-tuning, not pretraining.** LoRA trains a small number of adapter
  parameters against a frozen 1.5B base model. It is not training DeepSeek V4.1
  from scratch, and it is not a substitute for a frontier pretraining run.
- **Free T4 reality.** 16 GB VRAM, fp16 only (no native bf16), shared and
  time-limited. A 1.5B QLoRA run with 1024-token sequences is the comfortable
  case. A 3B run needs shorter sequences or a larger paid GPU.
- **Data quality matters more than epochs.** Start with 100-1000 clean rows and
  one epoch. Overfitting on a tiny set is easy.
- **Save early.** Free runtimes are deleted. Download the adapter or push it to
  the Hub before the session ends. `train-metrics.json` records what ran.
- **Secrets.** `HF_TOKEN` comes from notebook secrets or the environment. Never
  commit it, paste it into a cell, or print it. The optional push cell reads it
  at runtime and nothing else.
- **Licence and terms.** Check the base model licence and the dataset licence
  before publishing an adapter or merged model.
